In [1]:
import subprocess
subprocess.run(['pip', 'install', 'mediapy', 'flask', 'pyngrok', '-q'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], capture_output=True)

import io, os, threading
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import mediapy as media
from flask import Flask, request, send_file, jsonify
from pyngrok import ngrok

_UINT8_MAX_F = float(np.iinfo(np.uint8).max)

def load_image_from_bytes(data):
    image = tf.io.decode_image(data, channels=3)
    return tf.cast(image, dtype=tf.float32).numpy() / _UINT8_MAX_F

def _pad_to_align(x, align):
    height, width = x.shape[-3:-1]
    h_pad = (align - height % align) if height % align != 0 else 0
    w_pad = (align - width  % align) if width  % align != 0 else 0
    bbox_pad  = dict(offset_height=h_pad//2, offset_width=w_pad//2,
                     target_height=height+h_pad, target_width=width+w_pad)
    bbox_crop = dict(offset_height=h_pad//2, offset_width=w_pad//2,
                     target_height=height, target_width=width)
    return tf.image.pad_to_bounding_box(x, **bbox_pad), bbox_crop

class Interpolator:
    def __init__(self, align=64):
        self._model = hub.load('https://tfhub.dev/google/film/1')
        self._align = align
    def __call__(self, x0, x1, dt):
        if self._align:
            x0, bbox_to_crop = _pad_to_align(x0, self._align)
            x1, _ = _pad_to_align(x1, self._align)
        result = self._model({'x0': x0, 'x1': x1, 'time': dt[..., np.newaxis]}, training=False)
        image = result['image']
        if self._align:
            image = tf.image.crop_to_bounding_box(image, **bbox_crop)
        return image.numpy()

def _recursive_generator(f1, f2, n, interp):
    if n == 0:
        yield f1
    else:
        t = np.full((1,), 0.5, dtype=np.float32)
        mid = interp(np.expand_dims(f1, 0), np.expand_dims(f2, 0), t)[0]
        yield from _recursive_generator(f1, mid, n-1, interp)
        yield from _recursive_generator(mid, f2, n-1, interp)

def interpolate_recursively(frames, n, interp):
    for i in range(1, len(frames)):
        yield from _recursive_generator(frames[i-1], frames[i], n, interp)
    yield frames[-1]

print('Loading FILM model...')
interpolator = Interpolator()
print('Model ready.')

app = Flask(__name__)

@app.route('/run', methods=['POST'])
def run():
    f1_bytes = request.files['frame1'].read()
    f2_bytes = request.files['frame2'].read()
    img1 = load_image_from_bytes(f1_bytes)
    img2 = load_image_from_bytes(f2_bytes)
    times = int(request.form.get('times_to_interpolate', 6))
    fps   = int(request.form.get('fps', 30))
    frames = list(interpolate_recursively([img1, img2], times, interpolator))
    out_path = '/content/output.mp4'
    media.write_video(out_path, frames, fps=fps)
    return send_file(out_path, mimetype='video/mp4', as_attachment=True,
                     download_name='interpolated.mp4')

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok'})

public_url = ngrok.connect(5000)
print(f'Colab tunnel URL: {public_url}')

threading.Thread(target=lambda: app.run(port=5000)).start()


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Loading FILM model...
Model ready.


ERROR:pyngrok.process.ngrok:t=2026-10-11T13:13:03+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-10-11T13:13:03+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-10-11T13:13:03+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

PyngrokNgrokError: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.